In [ ]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt

import sys
sys.path.append('../')
from core import LADTransferTreeBoost, LSTransferTreeBoost
from utils import * #only needed for xgboost
from friedman1 import *

In [ ]:
#Function that returns best params for each pair of (d, target_instance) 
# and their average val rmse, rmse, mae, given a set of config columns (depending on method)
# This funcion can take a dataframe or a path to a csv as input.
# the best settings are chosen using the lowest val rmse
def topk_per_d_per_method(data, config_cols, k=5):
    """
    Return up to the top-k configs per group, ranked by avg_rmse.
    Averages are computed across seeds.
    """
    df = pd.read_csv(data, index_col = [0]) if isinstance(data, str) else data.copy()
    df = df.sort_values(by = ['seed'])

    agg = (df.groupby(config_cols, as_index=False)
            .agg(avg_rmse=('val_rmse','mean'),
                avg_mae=('val_mae','mean'),
                n_seeds=('seed','nunique')))
    agg = agg.merge(df, how = 'right')
    agg = agg.sort_values(by = 'avg_rmse')
    topk_df = agg.groupby(['method'], group_keys=False).head(k)
    topk_d = topk_df[['method', 'avg_rmse', 'rmse', 'mae']]
    return topk_d, topk_df




In [ ]:
# Function that returns best params for each pair of (d, target_instance) 
# and their average val rmse, rmse, mae, given a set of config columns (depending on method)
# This funcion can take a dataframe or a path to a csv as input.
# the best settings are chosen using the lowest val mae NOTE: different from previous method
# This func was used to choose optimal hyperparams in terms of lowest mae.

def topk_per_d_per_method_lad(data, config_cols, k=5):
    """
    Return up to the top-k configs per group, ranked by avg_rmse.
    Averages are computed across seeds.
    """
    df = pd.read_csv(data, index_col = [0]) if isinstance(data, str) else data.copy()
    df = df.sort_values(by = ['seed'])

    agg = (df.groupby(config_cols, as_index=False)
            .agg(avg_rmse=('val_rmse','mean'),
                avg_mae=('val_mae','mean'),
                n_seeds=('seed','nunique')))
    agg = agg.merge(df, how = 'right')
    agg = agg.sort_values(by = 'avg_mae')
    topk_df = agg.groupby(['method'], group_keys=False).head(k)
    topk_d = topk_df[['method', 'avg_rmse', 'rmse', 'mae']]
    return topk_d, topk_df

In [ ]:
# For gaussian errors, we simply combine previous results with LAD results
data_ls_gaussian = pd.read_csv('results/LSTransferTreeBoost_ablation_friedman.csv')
data = pd.read_csv('results/200_gaussian_6.csv')
data_ls_gaussian = data_ls_gaussian[(data_ls_gaussian['d'] == 6) & (data_ls_gaussian['target_instances'] == 200)]
data_ls_gaussian = data_ls_gaussian.drop(columns = ['target_instances', 'd'])
data_ls_gaussian

data = pd.concat([data, data_ls_gaussian])
data

# For slash errors
data_slash = pd.read_csv('results/200_slash_6.csv')

In [ ]:
# Make more runs with optimal settings for gaussian
seed_list = list(range(8,28))

# Here we find the optimal settings
LS_data = data[data['method'] == 'LSTransferTreeBoost']
LAD_data = data[data['method'] == 'LADTransferTreeBoost']

best_LS, best_LS_params = topk_per_d_per_method(
    LS_data,
['v', 'source_tree_size', 'target_tree_size', 'm_0', 'k'], k=5
)

best_LAD, best_LAD_params = topk_per_d_per_method(
    LAD_data,
['v', 'source_tree_size', 'target_tree_size', 'm_0', 'k'], k=5
)

df = pd.DataFrame(columns = ['seed', 'method', 'rmse', 'mae'])

for seed in seed_list:

    
    X_target_test, y_target_test = friedman1(n_samples=1000, add_noise = False, noise_distribution = 'gaussian', n_features=10, random_seed=seed) #do NOT add noise to test set!!!!
    X_target_val, y_target_val = friedman1(n_samples=1000, add_noise = False, noise_distribution = 'gaussian', n_features=10, random_seed=seed + 10) #do NOT add noise to test set!!!!
    X_target_train, y_target_train = friedman1(n_samples=200, add_noise = True, noise_distribution = 'gaussian', n_features=10, random_seed=seed) #add noise to train set
    X_source_train, y_source_train = friedman1_altered(n_samples=1000, add_noise = True, noise_distribution = 'gaussian',
                                                    n_features=10, d=6, shift_seed=seed, random_seed = seed) #also add noise to source (only train here)
    



    # Test for all methods with optimal settings!!!!

    v = best_LS_params['v'].values[0]
    source_tree_size = best_LS_params['source_tree_size'].values[0]
    target_tree_size = best_LS_params['target_tree_size'].values[0]
    m_0 = best_LS_params['m_0'].values[0]
    k = best_LS_params['k'].values[0]
    method = f'LSTransferTreeBoost'
    fiter = LSTransferTreeBoost(epochs=1000, v=v, source_tree_size=source_tree_size, 
                            target_tree_size=target_tree_size, k=k, m_0=m_0)
    fiter.fit(X_target_train, y_target_train, X_source_train, y_source_train, val_x=X_target_val, val_y=y_target_val, early_stopping_rounds=8, show_curves=False)
    rmse = fiter.evaluate(X_target_test, y_target_test, metric = 'rmse')
    val_rmse = fiter.evaluate(X_target_val, y_target_val, metric = 'rmse')
    mae = fiter.evaluate(X_target_test, y_target_test, metric = 'mae')
    val_mae = fiter.evaluate(X_target_val, y_target_val, metric = 'mae')
    df.loc[len(df)] = [seed, method, rmse, mae] 
    

    v = best_LAD_params['v'].values[0]
    source_tree_size = best_LAD_params['source_tree_size'].values[0]
    target_tree_size = best_LAD_params['target_tree_size'].values[0]
    m_0 = best_LAD_params['m_0'].values[0]
    k = best_LAD_params['k'].values[0]
    method = f'LADTransferTreeBoost'
    fiter = LADTransferTreeBoost(epochs=1000, v=v, source_tree_size=source_tree_size, 
                            target_tree_size=target_tree_size, k=k, m_0=m_0)
    fiter.fit(X_target_train, y_target_train, X_source_train, y_source_train, val_x=X_target_val, val_y=y_target_val, early_stopping_rounds=8, show_curves=False)
    rmse = fiter.evaluate(X_target_test, y_target_test, metric = 'rmse')
    val_rmse = fiter.evaluate(X_target_val, y_target_val, metric = 'rmse')
    mae = fiter.evaluate(X_target_test, y_target_test, metric = 'mae')
    val_mae = fiter.evaluate(X_target_val, y_target_val, metric = 'mae')
    df.loc[len(df)] = [seed, method, rmse, mae] 
    df.to_csv('results/200_gaussian_6_optim.csv')

In [ ]:
#Make more runs with optimal settings for Slash
seed_list = list(range(8,28))


LS_data = data_slash[data_slash['method'] == 'LSTransferTreeBoost']
LAD_data = data_slash[data_slash['method'] == 'LADTransferTreeBoost']

#Fins optimal settings
best_LS, best_LS_params = topk_per_d_per_method(
    LS_data,
['v', 'source_tree_size', 'target_tree_size', 'm_0', 'k'], k=5
)

best_LAD, best_LAD_params = topk_per_d_per_method(
    LAD_data,
['v', 'source_tree_size', 'target_tree_size', 'm_0', 'k'], k=5
)

df = pd.DataFrame(columns = ['seed', 'method', 'rmse', 'mae'])

for seed in seed_list:

    
    X_target_test, y_target_test = friedman1(n_samples=1000, add_noise = False, noise_distribution = 'slash', n_features=10, random_seed=seed) #do NOT add noise to test set!!!!
    X_target_val, y_target_val = friedman1(n_samples=1000, add_noise = False, noise_distribution = 'slash', n_features=10, random_seed=seed + 10) #do NOT add noise to test set!!!!
    X_target_train, y_target_train = friedman1(n_samples=200, add_noise = True, noise_distribution = 'slash', n_features=10, random_seed=seed) #add noise to train set
    X_source_train, y_source_train = friedman1_altered(n_samples=1000, add_noise = True, noise_distribution = 'slash',
                                                    n_features=10, d=6, shift_seed=seed, random_seed = seed) #also add noise to source (only train here)
    



    #Test for all methods with optimal settings

    v = best_LS_params['v'].values[0]
    source_tree_size = best_LS_params['source_tree_size'].values[0]
    target_tree_size = best_LS_params['target_tree_size'].values[0]
    m_0 = best_LS_params['m_0'].values[0]
    k = best_LS_params['k'].values[0]
    method = f'LSTransferTreeBoost'
    fiter = LSTransferTreeBoost(epochs=1000, v=v, source_tree_size=source_tree_size, 
                            target_tree_size=target_tree_size, k=k, m_0=m_0)
    fiter.fit(X_target_train, y_target_train, X_source_train, y_source_train, val_x=X_target_val, val_y=y_target_val, early_stopping_rounds=8, show_curves=False)
    rmse = fiter.evaluate(X_target_test, y_target_test, metric = 'rmse')
    val_rmse = fiter.evaluate(X_target_val, y_target_val, metric = 'rmse')
    mae = fiter.evaluate(X_target_test, y_target_test, metric = 'mae')
    val_mae = fiter.evaluate(X_target_val, y_target_val, metric = 'mae')
    df.loc[len(df)] = [seed, method, rmse, mae] 
    

    v = best_LAD_params['v'].values[0]
    source_tree_size = best_LAD_params['source_tree_size'].values[0]
    target_tree_size = best_LAD_params['target_tree_size'].values[0]
    m_0 = best_LAD_params['m_0'].values[0]
    k = best_LAD_params['k'].values[0]
    method = f'LADTransferTreeBoost'
    fiter = LADTransferTreeBoost(epochs=1000, v=v, source_tree_size=source_tree_size, 
                            target_tree_size=target_tree_size, k=k, m_0=m_0)
    fiter.fit(X_target_train, y_target_train, X_source_train, y_source_train, val_x=X_target_val, val_y=y_target_val, early_stopping_rounds=8, show_curves=False)
    rmse = fiter.evaluate(X_target_test, y_target_test, metric = 'rmse')
    val_rmse = fiter.evaluate(X_target_val, y_target_val, metric = 'rmse')
    mae = fiter.evaluate(X_target_test, y_target_test, metric = 'mae')
    val_mae = fiter.evaluate(X_target_val, y_target_val, metric = 'mae')
    df.loc[len(df)] = [seed, method, rmse, mae] 
                                                        
    df.to_csv('results/200_slash_6_optim.csv')

In [ ]:
# Final viz comes here!!!!
data = pd.read_csv('results/200_gaussian_6_optim.csv')
data_slash = pd.read_csv('results/200_slash_6_optim.csv')
# We remove one very weird result. As this did not happen in any other experiment, 
# we believe this is a numerical issue with the code. We remove the seed for both methods.
data = data[data['seed'] != 18] 
data_slash = data_slash[data_slash['seed'] != 18]
data['method'] = data['method'].replace("LSTransferTreeBoost", "LS")
data['method'] = data['method'].replace("LADTransferTreeBoost", "LAD")

data_slash['method'] = data_slash['method'].replace("LSTransferTreeBoost", "LS")
data_slash['method'] = data_slash['method'].replace("LADTransferTreeBoost", "LAD")

plt.figure(figsize = (14,10))

plt.subplot(2,2,1)
sns.boxplot(data = data, x = 'method', y = 'rmse')
plt.gca().set_xlabel(None)
plt.gca().set_ylabel('RMSE', fontsize=14)
plt.gca().set_title('Gaussian', fontsize=16)

plt.subplot(2,2,2)
sns.boxplot(data = data, x = 'method', y = 'mae')
plt.gca().set_xlabel(None)
plt.gca().set_ylabel('MAE', fontsize=14)
plt.gca().set_title('Gaussian', fontsize=16)

plt.subplot(2,2,3)
sns.boxplot(data = data_slash, x = 'method', y = 'rmse')
plt.gca().set_xlabel(None)
plt.gca().set_ylabel('RMSE', fontsize=14)
plt.gca().set_title('Slash', fontsize=16)

plt.subplot(2,2,4)
sns.boxplot(data = data_slash, x = 'method', y = 'mae')
plt.gca().set_xlabel(None)
plt.gca().set_ylabel('MAE', fontsize=14)
plt.gca().set_title('Slash', fontsize=16)

plt.savefig('vizes/ls-lad.png', dpi = 300, bbox_inches = 'tight', pad_inches = 0.1)





